# jB_02 - Mengapa adversarial training merusak UNSW? (JURNAL)

Di ISRITI (R6) kami melaporkan temuan yang jujur tetapi **belum diselidiki**: adversarial
training (FGSM, 50% sampel) menaikkan robustness pada CIC tetapi **secara konsisten
merusak UNSW** di kelima seed (clean MCC 0,61->0,17; recall 0,70->0,10). Notebook ini
**menyelidiki penyebabnya** dari artefak yang sudah ada (tidak perlu run EC2 baru):
model clean `run_S_R3*` dan adv-trained `run_S_R6*` + partisi klien dari s01.

**Hipotesis yang diuji:**
- H1 **Recall collapse / benign bias**: model adv-trained memprediksi hampir semua
  aliran UNSW sebagai benign (confusion matrix per domain).
- H2 **Pergeseran skor keputusan**: distribusi skor sigmoid UNSW bergeser ke bawah 0,5
  setelah adv-training (histogram skor clean vs adv, per domain).
- H3 **Konflik arah pembaruan antar-domain**: contoh FGSM dibuat di z-space LOKAL tiap
  klien; karena rasio serangan & skala UNSW vs CIC berbeda, perturbasi menggeser batas
  keputusan dgn arah yang tak sejalan. Diukur via pergeseran rata-rata skor benign vs
  attack per domain (clean->adv).
- H4 **Interaksi rasio serangan**: UNSW train 55% attack vs CIC 17%; mengganti 50%
  sampel dgn adversarial example berdampak beda. Dilaporkan sebagai konteks.

**Output:** `journal_out/jB2_unsw_diagnosis.{json,csv}` + figure
`figure-fed/fig_unsw_degradation.png` (upload S3). Rata-rata lima seed.

> Jalankan di SageMaker. Angka dari eksekusi nyata. Lapor apa adanya walau tak tuntas.

## Sel 1 - Setup + muat partisi + model clean/adv (5 seed)

In [ ]:
import importlib, sys, subprocess
for m in ('numpy','scikit-learn','boto3','matplotlib','scipy'):
    if importlib.util.find_spec(m) is None:
        subprocess.run([sys.executable,'-m','pip','install','-q',m],check=True)
import os, io, json, csv, numpy as np, boto3
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt

BUCKET='ssh-detection-features-232032302717'; REGION='ap-southeast-1'
BASE='unsw-far/federated_survey'           # data & model survei (far-domain) ada di sini
OUTDIR='journal_out'; os.makedirs(OUTDIR,exist_ok=True)
FIGDIR='../../figure-fed'                    # relatif notebook; fallback ke OUTDIR bila tak ada
if not os.path.isdir(FIGDIR): FIGDIR=OUTDIR
ADV_EPS=float(os.environ.get('ADV_EPS','0.1'))
SEEDS=[13,42,101,202,303]
s3=boto3.client('s3',region_name=REGION)
def load_npz_s3(key):
    try: obj=s3.get_object(Bucket=BUCKET,Key=key)
    except Exception as e: print('MISS',key,e); return None
    return np.load(io.BytesIO(obj['Body'].read()),allow_pickle=True)
def run_name(base,seed): return base if seed==42 else f'{base}_s{seed}'

# partisi klien + z-space LOKAL (statistik train klien; sama spt training EC2)
P={}
for nm in ['cic_train','cic_test','unsw_train','unsw_test']:
    sub='cic' if nm.startswith('cic') else 'unsw'
    d=load_npz_s3(f'{BASE}/data/{sub}/{nm}.npz'); assert d is not None,'jalankan s01 dulu'
    P[nm]=(np.asarray(d['X'],float),np.asarray(d['y'],int))
CL=['cic','unsw']; Z={}
for c in CL:
    Xtr=P[f'{c}_train'][0]; m=Xtr.mean(0); s=Xtr.std(0); s[s==0]=1
    Z[c]=dict(Xtr=(Xtr-m)/s, ytr=P[f'{c}_train'][1],
              Xte=(P[f'{c}_test'][0]-m)/s, yte=P[f'{c}_test'][1],
              lo=np.percentile((Xtr-m)/s,1,axis=0), hi=np.percentile((Xtr-m)/s,99,axis=0))
    print(c,'train',Z[c]['Xtr'].shape,'pos=%.3f'%Z[c]['ytr'].mean(),
          '| test pos=%.3f'%Z[c]['yte'].mean())
print('SEEDS=',SEEDS,'ADV_EPS=',ADV_EPS)

## Sel 2 - MLP forward + util (prob skor, confusion, PGD)

In [ ]:
relu=lambda x:np.maximum(0,x); sig=lambda x:1/(1+np.exp(-np.clip(x,-30,30)))
def fwd(W,X):
    a1=X@W[0]+W[1]; z1=relu(a1); a2=z1@W[2]+W[3]; z2=relu(a2); a3=z2@W[4]+W[5]
    return sig(a3).ravel(),(a1,z1,a2,z2)
def prob(W,X): o,_=fwd(W,X); return o
def pred(W,X): return (prob(W,X)>=0.5).astype(int)
def dX(W,X,y):
    o,(a1,z1,a2,z2)=fwd(W,X); g=(o-y.astype(float))[:,None]
    d2=(g@W[4].T)*(a2>0); d1=(d2@W[2].T)*(a1>0); return d1@W[0].T
def pgd(W,X,y,eps,lo,hi,steps=10):
    a=eps/4; Xa=X.copy()
    for _ in range(steps):
        Xa=np.clip(np.clip(Xa+a*np.sign(dX(W,Xa,y)),X-eps,X+eps),lo,hi)
    return Xa
def confusion(y,p):
    y=y.astype(int); p=p.astype(int)
    return dict(tp=int(((p==1)&(y==1)).sum()),tn=int(((p==0)&(y==0)).sum()),
                fp=int(((p==1)&(y==0)).sum()),fn=int(((p==0)&(y==1)).sum()))
def load_model(run):
    gf=load_npz_s3(f'{BASE}/run_{run}/global_final.npz')
    return [gf[f'w{i}'] for i in range(6)] if gf is not None else None
print('util siap')

## Sel 3 - Diagnosis per seed: H1 (confusion) + H2/H3 (skor) per domain

In [ ]:
def diagnose(W):
    """Kembalikan metrik per domain: confusion clean, rata-rata skor benign/attack,
    fraksi prediksi-attack (untuk deteksi benign-bias)."""
    out={}
    for c in CL:
        X,y=Z[c]['Xte'],Z[c]['yte']; o=prob(W,X); p=(o>=0.5).astype(int)
        cm=confusion(y,p)
        out[c]=dict(**cm,
                    frac_pred_attack=float(p.mean()),
                    mean_score_benign=float(o[y==0].mean()),
                    mean_score_attack=float(o[y==1].mean()),
                    recall=cm['tp']/max(cm['tp']+cm['fn'],1))
    return out

# kumpulkan per seed: clean vs adv
diag={c:{'clean':[], 'adv':[]} for c in CL}
score_hist={c:{'clean':None,'adv':None,'y':None} for c in CL}  # simpan skor seed 42 utk histogram
for seed in SEEDS:
    Wc=load_model(run_name('S_R3',seed)); Wa=load_model(run_name('S_R6',seed))
    if Wc is None or Wa is None: print('[lewati] seed',seed,'model belum ada'); continue
    dc=diagnose(Wc); da=diagnose(Wa)
    for c in CL:
        diag[c]['clean'].append(dc[c]); diag[c]['adv'].append(da[c])
    if seed==42:  # simpan distribusi skor satu seed utk figure
        for c in CL:
            X,y=Z[c]['Xte'],Z[c]['yte']
            score_hist[c]={'clean':prob(Wc,X),'adv':prob(Wa,X),'y':y}
    uc=diag; r_c=dc['unsw']['recall']; r_a=da['unsw']['recall']
    print(f'seed {seed:<3} UNSW recall clean={r_c:.3f} -> adv={r_a:.3f} | '
          f'frac_pred_attack clean={dc["unsw"]["frac_pred_attack"]:.3f} -> adv={da["unsw"]["frac_pred_attack"]:.3f}')
print('selesai diagnosis per seed')

## Sel 4 - Ringkas antar-seed + simpan JSON/CSV

In [ ]:
def avg(dicts,key): return float(np.mean([d[key] for d in dicts]))
def sd(dicts,key): return float(np.std([d[key] for d in dicts],ddof=1)) if len(dicts)>1 else 0.0

summary={'eps':ADV_EPS,'seeds_used':[]}
rows=[]
for c in CL:
    cl=diag[c]['clean']; ad=diag[c]['adv']; n=len(cl)
    summary['seeds_used']=SEEDS[:n]
    block={}
    for key in ['recall','frac_pred_attack','mean_score_benign','mean_score_attack']:
        block[key]={'clean_mean':round(avg(cl,key),4),'clean_sd':round(sd(cl,key),4),
                    'adv_mean':round(avg(ad,key),4),'adv_sd':round(sd(ad,key),4),
                    'delta':round(avg(ad,key)-avg(cl,key),4)}
        rows.append({'domain':c,'metric':key,'clean':block[key]['clean_mean'],
                     'adv':block[key]['adv_mean'],'delta':block[key]['delta']})
    summary[c]=block

json.dump(summary,open(os.path.join(OUTDIR,'jB2_unsw_diagnosis.json'),'w'),indent=2)
with open(os.path.join(OUTDIR,'jB2_unsw_diagnosis.csv'),'w',newline='') as f:
    w=csv.DictWriter(f,fieldnames=['domain','metric','clean','adv','delta']); w.writeheader(); w.writerows(rows)

print('=== RINGKAS (mean 5 seed) ===')
for c in CL:
    b=summary[c]
    print(f'[{c.upper()}] recall {b["recall"]["clean_mean"]:.3f}->{b["recall"]["adv_mean"]:.3f} '
          f'| frac_pred_attack {b["frac_pred_attack"]["clean_mean"]:.3f}->{b["frac_pred_attack"]["adv_mean"]:.3f} '
          f'| skor benign {b["mean_score_benign"]["clean_mean"]:.3f}->{b["mean_score_benign"]["adv_mean"]:.3f} '
          f'| skor attack {b["mean_score_attack"]["clean_mean"]:.3f}->{b["mean_score_attack"]["adv_mean"]:.3f}')
print('tersimpan jB2_unsw_diagnosis.{json,csv}')

## Sel 5 - Figure: pergeseran skor keputusan (histogram) CIC vs UNSW

In [ ]:
plt.rcParams.update({'font.size':9,'figure.dpi':160,'savefig.bbox':'tight'})
fig,axes=plt.subplots(2,2,figsize=(8.2,5.0),sharex=True)
for col,c in enumerate(CL):
    sh=score_hist[c]
    if sh['clean'] is None: continue
    y=sh['y']
    for row,(tag,sc) in enumerate([('clean',sh['clean']),('adv-trained',sh['adv'])]):
        ax=axes[row][col]
        ax.hist(sc[y==0],bins=40,range=(0,1),alpha=0.6,color='#3a6ea5',label='benign',density=True)
        ax.hist(sc[y==1],bins=40,range=(0,1),alpha=0.6,color='#d1495b',label='attack',density=True)
        ax.axvline(0.5,color='#333',lw=1,ls='--')
        ax.set_title(f'{c.upper()} - {tag}',fontsize=9)
        if row==1: ax.set_xlabel('model score (>=0.5 -> attack)',fontsize=8)
        if col==0: ax.set_ylabel('density',fontsize=8)
        if row==0 and col==1: ax.legend(fontsize=7,loc='upper center')
fig.suptitle('Decision-score distribution: clean vs adversarially trained (seed 42)',fontsize=10)
fig.tight_layout()
fp=os.path.join(FIGDIR,'fig_unsw_degradation.png'); fig.savefig(fp); plt.close(fig)
print('saved',fp)
try:
    for f in ['jB2_unsw_diagnosis.json','jB2_unsw_diagnosis.csv']:
        s3.upload_file(os.path.join(OUTDIR,f),BUCKET,f'{BASE}/results/{f}')
    s3.upload_file(fp,BUCKET,f'{BASE}/results/fig_unsw_degradation.png')
    print('upload hasil + figure ke S3 OK')
except Exception as e: print('(upload dilewati:',e,')')
print('=== jB_02 SELESAI ===')

## Catatan untuk naskah jurnal

- Tujuan: menjawab pertanyaan terbuka ISRITI (sebab degradasi UNSW), bukan menambah klaim.
- H1 recall collapse / benign-bias terukur via `frac_pred_attack` dan recall UNSW.
- H2/H3 via pergeseran rata-rata skor benign/attack dan histogram skor (Figure).
  Jika skor attack UNSW anjlok ke bawah 0,5 setelah adv-training, itu bukti model
  menjadi benign-bias pada domain UNSW.
- H4 (rasio serangan 55% vs 17%) dilaporkan sebagai konteks penjelas.
- Jika penyebab tak tuntas satu faktor, laporkan kombinasi faktor secara jujur.